# Week 4: Web Scraping + Networked Archives
### Google Colab version

This notebook introduces web scraping as both a **technical method** and a **critical archival practice**.

---
**What you'll do**
- define a small scraping question
- collect a tiny dataset from a public webpage
- save a links dataset
- save an image dataset
- save a sentence dataset using keywords
- combine all datasets into one larger dataset
- generate a simple webpage to visualize the dataset

**Course connection**
- **Week 1:** erosion  
- **Week 2:** systems and dependencies  
- **Week 3:** fragments and incomplete provenance  
- **Week 4:** scraping as a way of producing new fragments

---
## 1. Framing

**Web scraping** means extracting material from a webpage, often:
- text
- links
- images
- metadata

But scraping is **never neutral**.  
It removes material from its original:
- interface
- chronology
- context
- platform logic

A scraped dataset is **not the internet itself**.  
It is a **partial reconstruction produced by a method**.

> “Found images become like textures or pigments… shorn of their historical specificity.”  
> — Ian Rothwell

Use this idea as a guide while you work: scraping does not just collect material, it **transforms** it.

---
## 2. Setup

Google Colab already runs Python in the browser, so you do **not** need to install anything locally.

Run the next two cells.

In [2]:
# Install libraries if needed
!pip -q install beautifulsoup4 pandas requests

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urljoin, urlparse
from IPython.display import display, HTML, FileLink
import urllib.robotparser
import re
import time
from datetime import datetime, timezone

---
## 3. Choose one or more source pages

For this demo, we use a short list of public pages.

You can later replace these with pages that:
- are publicly accessible
- do not require login
- are appropriate for classroom use
- allow respectful access

**Avoid** scraping private, sensitive, or restricted material.

Start small: 3–5 pages is enough for most beginner projects.

---
## 3b. Before you fetch anything — check robots.txt

Every website can publish a `robots.txt` file that tells automated tools
what they are and are not allowed to access.

This is the same mechanism the New York Times used to block the Internet Archive's crawlers.
The identity of your scraper is politically legible to the server,
even when it is invisible to you.

Run this cell before fetching any pages.

In [5]:
def check_robots(url):
    parsed = urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = urllib.robotparser.RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
        allowed = rp.can_fetch("*", url)
        print(f"URL: {url}")
        print(f"  robots.txt: {robots_url}")
        print(f"  scraping allowed: {allowed}")
        print()
    except Exception as e:
        print(f"{url} — could not read robots.txt: {e}")

# Define your URLs here — robots check runs before any fetching
urls = [
    "https://archive.org",
    "https://en.wikipedia.org/wiki/Internet_Archive"
]

for url in urls:
    check_robots(url)

# Note: the Internet Archive's robots.txt recently changed from
# 'Welcome to the Archive! Please crawl our files.'
# to simply 'Welcome to the Internet Archive!'
# The open invitation to crawlers is gone.

URL: https://archive.org
  robots.txt: https://archive.org/robots.txt
  scraping allowed: True

URL: https://en.wikipedia.org/wiki/Internet_Archive
  robots.txt: https://en.wikipedia.org/robots.txt
  scraping allowed: False



In [6]:
pages = []
failed = []

for url in urls:
    try:
        response = requests.get(url, timeout=20, headers={
            # This User-Agent string identifies your scraper to the server.
            # Archivarius uses 'ArchivariusBot/1.0 (+contact-url)'.
            # The NYT blocks 'archive.org_bot' by name in its robots.txt.
            # The scraper's identity is visible to the server even when invisible to you.
            "User-Agent": "Mozilla/5.0 (educational scraping project)"
        })
        response.raise_for_status()
        pages.append({
            "url": url,
            "html": response.text,
            "soup": BeautifulSoup(response.text, "html.parser"),
            "fetched_at": datetime.now(timezone.utc).isoformat(),
            "status_code": response.status_code
        })
        print(f"checkmark {url} — status {response.status_code}")
    except Exception as e:
        failed.append({"url": url, "error": str(e)})
        print(f"failed {url} — {e}")

if failed:
    print(f"\n{len(failed)} page(s) could not be fetched:")
    for f in failed:
        print(f"  {f['url']}: {f['error']}")

checkmark https://archive.org — status 200
checkmark https://en.wikipedia.org/wiki/Internet_Archive — status 200


---
## 3d. Look at the raw HTML before extracting anything

This is what the scraper actually sees.
No colors. No layout. No images rendered. No hover states. No navigation.
Just text and tags.

Everything you collect from this is a decision you make.

In [ ]:
print("Raw HTML — first 800 characters of page 1:\n")
print(pages[0]["html"][:800])
print("\n---")
print("What you see above is what the scraper sees.")
print("The webpage you visited in a browser is an interpretation of this.")
print("Your dataset will be built from this raw text, not from the page as you experienced it.")

---
## 4. Parse the pages
**Small glossary**:

*   **Crawl** - The process of discovering URLs from media sources using various strategies (sitemaps, RSS feeds, APIs, etc.)

*   **Parse** - The process of extracting structured data (title, author, content, metadata) from raw scraped content.

HTML is the structure underneath a webpage.

A basic multi-page scraping workflow usually looks like this:

1. define a list of URLs  
2. request each page  
3. parse the HTML  
4. locate elements on each page  
5. extract data into shared datasets  
6. save those datasets in a new form

In [7]:
print("Page titles:")
for page in pages:
    title = page["soup"].title.get_text(strip=True) if page["soup"].title else "[no title]"
    print("-", title)

Page titles:
- Internet Archive: Digital Library of Free & Borrowable Texts, Movies, Music & Wayback Machine
- Internet Archive - Wikipedia


---
## 5. Extract links from multiple pages

Links make a good beginner dataset because they are easy to inspect.

In the next cell, we collect from **all pages**:
- visible link text
- original href
- absolute URL
- domain
- source page

In [8]:
link_rows = []

for page in pages:
    source_url = page["url"]
    soup = page["soup"]

    for a in soup.find_all("a"):
        href = a.get("href")
        text = a.get_text(" ", strip=True)
        absolute_url = urljoin(source_url, href) if href else None
        link_rows.append({
            "item_type": "link",
            "label": text,
            "href": href,
            "absolute_url": absolute_url,
            "domain": urlparse(absolute_url).netloc if absolute_url else None,
            "source_page": source_url,
            "fetched_at": page.get("fetched_at"),
            "thumbnail_url": None
        })

links_df = pd.DataFrame(link_rows)
links_df.head(10)

,item_type,label,href,absolute_url,domain,source_page,fetched_at,thumbnail_url
0,link,Jump to content,#bodyContent,https://en.wikipedia.org/wiki/Internet_Archive...,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None
1,link,Main page,/wiki/Main_Page,https://en.wikipedia.org/wiki/Main_Page,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None
2,link,Contents,/wiki/Wikipedia:Contents,https://en.wikipedia.org/wiki/Wikipedia:Contents,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None
3,link,Current events,/wiki/Portal:Current_events,https://en.wikipedia.org/wiki/Portal:Current_e...,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None
4,link,Random article,/wiki/Special:Random,https://en.wikipedia.org/wiki/Special:Random,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None
5,link,About Wikipedia,/wiki/Wikipedia:About,https://en.wikipedia.org/wiki/Wikipedia:About,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None
6,link,Contact us,//en.wikipedia.org/wiki/Wikipedia:Contact_us,https://en.wikipedia.org/wiki/Wikipedia:Contac...,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None
7,link,Help,/wiki/Help:Contents,https://en.wikipedia.org/wiki/Help:Contents,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None
8,link,Learn to edit,/wiki/Help:Introduction,https://en.wikipedia.org/wiki/Help:Introduction,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None
9,link,Community portal,/wiki/Wikipedia:Community_portal,https://en.wikipedia.org/wiki/Wikipedia:Commun...,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None


---
## 6. Save the links dataset

In [ ]:
links_csv = "week4_links_dataset.csv"
links_df.to_csv(links_csv, index=False)
print(f"Saved: {links_csv}")

---
## 7. Extract images from multiple pages

Now we collect image information from **all pages**.

This will create a second dataset with:
- alt text
- image URL
- domain
- source page

Then we will combine it with the links dataset from above.

In [9]:
image_rows = []

for page in pages:
    source_url = page["url"]
    soup = page["soup"]

    for img in soup.find_all("img"):
        src = img.get("src")
        alt = img.get("alt", "")
        full_src = urljoin(source_url, src) if src else None
        image_rows.append({
            "item_type": "image",
            "label": alt if alt else "[no alt text]",
            "href": src,
            "absolute_url": full_src,
            "domain": urlparse(full_src).netloc if full_src else None,
            "source_page": source_url,
            "fetched_at": page.get("fetched_at"),
            "thumbnail_url": full_src
        })

images_df = pd.DataFrame(image_rows)
images_df.head(10)

,item_type,label,href,absolute_url,domain,source_page,fetched_at,thumbnail_url
0,image,[no alt text],/static/images/icons/enwiki-25.svg,https://en.wikipedia.org/static/images/icons/e...,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,https://en.wikipedia.org/static/images/icons/e...
1,image,Wikipedia,/static/images/mobile/copyright/wikipedia-word...,https://en.wikipedia.org/static/images/mobile/...,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,https://en.wikipedia.org/static/images/mobile/...
2,image,The Free Encyclopedia,/static/images/mobile/copyright/wikipedia-tagl...,https://en.wikipedia.org/static/images/mobile/...,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,https://en.wikipedia.org/static/images/mobile/...
3,image,Internet Archive logo,//upload.wikimedia.org/wikipedia/commons/thumb...,https://upload.wikimedia.org/wikipedia/commons...,upload.wikimedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,https://upload.wikimedia.org/wikipedia/commons...
4,image,Decrease,//upload.wikimedia.org/wikipedia/commons/thumb...,https://upload.wikimedia.org/wikipedia/commons...,upload.wikimedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,https://upload.wikimedia.org/wikipedia/commons...
5,image,Increase,//upload.wikimedia.org/wikipedia/commons/thumb...,https://upload.wikimedia.org/wikipedia/commons...,upload.wikimedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,https://upload.wikimedia.org/wikipedia/commons...
6,image,Decrease,//upload.wikimedia.org/wikipedia/commons/thumb...,https://upload.wikimedia.org/wikipedia/commons...,upload.wikimedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,https://upload.wikimedia.org/wikipedia/commons...
7,image,Edit this at Wikidata,//upload.wikimedia.org/wikipedia/en/thumb/8/8a...,https://upload.wikimedia.org/wikipedia/en/thum...,upload.wikimedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,https://upload.wikimedia.org/wikipedia/en/thum...
8,image,Tor network,//upload.wikimedia.org/wikipedia/commons/thumb...,https://upload.wikimedia.org/wikipedia/commons...,upload.wikimedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,https://upload.wikimedia.org/wikipedia/commons...
9,image,Edit this at Wikidata,//upload.wikimedia.org/wikipedia/en/thumb/8/8a...,https://upload.wikimedia.org/wikipedia/en/thum...,upload.wikimedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,https://upload.wikimedia.org/wikipedia/en/thum...


---
## 8. Save the image dataset

In [ ]:
images_csv = "week4_images_dataset.csv"
images_df.to_csv(images_csv, index=False)
print(f"Saved: {images_csv}")

---
## 9. Extract sentences from multiple pages using a list of keywords

This scraper lets you collect **sentences** from all selected pages based on keywords.

It will:
- pull paragraph text from each page
- split it into sentences
- search for sentences containing selected keywords
- save those sentences as a third dataset

This is useful if you want to treat **language as archive material**, not just links or images.

In [10]:
import re

# Edit this list for your own project
keywords = ["example", "domain", "use"]

sentence_rows = []

for page in pages:
    source_url = page["url"]
    soup = page["soup"]

    paragraph_text = " ".join(
        p.get_text(" ", strip=True) for p in soup.find_all("p")
    )

    raw_sentences = re.split(r'(?<=[.!?])\s+', paragraph_text)

    for sentence in raw_sentences:
        cleaned = sentence.strip()
        if not cleaned:
            continue

        matched_keywords = [kw for kw in keywords if kw.lower() in cleaned.lower()]

        if matched_keywords:
            sentence_rows.append({
                "item_type": "sentence",
                "label": cleaned,
                "href": None,
                "absolute_url": None,
                "domain": urlparse(source_url).netloc,
                "source_page": source_url,
                "thumbnail_url": None,
                "matched_keywords": ", ".join(matched_keywords)
            })

sentences_df = pd.DataFrame(sentence_rows)
sentences_df.head(20)

,item_type,label,href,absolute_url,domain,source_page,thumbnail_url,matched_keywords
0,sentence,The archive is used frequently by journalists ...,None,None,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,None,use
1,sentence,The announcement received widespread coverage ...,None,None,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,None,use
2,sentence,[ 37 ] All material is then digitized and reta...,None,None,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,None,domain
3,sentence,"[ 38 ] On June 1, 2020, four large publishing ...",None,None,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,None,use
4,sentence,"See 31 million of you on HIBP !"" [ 59 ] [ 55 ]...",None,None,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,None,use
5,sentence,[ 58 ] [ 60 ] The attackers stole users' email...,None,None,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,None,use
6,sentence,[ 72 ] It uses Ubuntu as its choice of operati...,None,None,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,None,use
7,sentence,[ 87 ] It can be used to see what previous ver...,None,None,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,None,use
8,sentence,[ 91 ] Archive-It allows the user to customize...,None,None,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,None,use
9,sentence,"As of 2018, Archive-it was partnering with mor...",None,None,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,None,use


---
## 10. Save the sentence dataset

In [ ]:
sentences_csv = "week4_sentences_dataset.csv"
sentences_df.to_csv(sentences_csv, index=False)
print(f"Saved: {sentences_csv}")

---
## 11. Add the new datasets to the preexisting datasets from before

We now combine:
- the links dataset from all pages
- the image dataset from all pages
- the sentence dataset from all pages based on keywords

This creates one larger dataset that can be visualized together.

In [14]:
combined_df = pd.concat([links_df, images_df, sentences_df], ignore_index=True)

combined_csv = "week4_combined_dataset.csv"
combined_df.to_csv(combined_csv, index=False)

print(f"Saved combined dataset: {combined_csv}")
print("Rows in links dataset:", len(links_df))
print("Rows in image dataset:", len(images_df))
print("Rows in sentence dataset:", len(sentences_df))
print("Rows in combined dataset:", len(combined_df))

combined_df.head(20)

Saved combined dataset: week4_combined_dataset.csv
Rows in links dataset: 2444
Rows in image dataset: 42
Rows in sentence dataset: 48
Rows in combined dataset: 2534


,item_type,label,href,absolute_url,domain,source_page,fetched_at,thumbnail_url,matched_keywords
0,link,Jump to content,#bodyContent,https://en.wikipedia.org/wiki/Internet_Archive...,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None,NaN
1,link,Main page,/wiki/Main_Page,https://en.wikipedia.org/wiki/Main_Page,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None,NaN
2,link,Contents,/wiki/Wikipedia:Contents,https://en.wikipedia.org/wiki/Wikipedia:Contents,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None,NaN
3,link,Current events,/wiki/Portal:Current_events,https://en.wikipedia.org/wiki/Portal:Current_e...,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None,NaN
4,link,Random article,/wiki/Special:Random,https://en.wikipedia.org/wiki/Special:Random,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None,NaN
5,link,About Wikipedia,/wiki/Wikipedia:About,https://en.wikipedia.org/wiki/Wikipedia:About,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None,NaN
6,link,Contact us,//en.wikipedia.org/wiki/Wikipedia:Contact_us,https://en.wikipedia.org/wiki/Wikipedia:Contac...,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None,NaN
7,link,Help,/wiki/Help:Contents,https://en.wikipedia.org/wiki/Help:Contents,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None,NaN
8,link,Learn to edit,/wiki/Help:Introduction,https://en.wikipedia.org/wiki/Help:Introduction,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None,NaN
9,link,Community portal,/wiki/Wikipedia:Community_portal,https://en.wikipedia.org/wiki/Wikipedia:Commun...,en.wikipedia.org,https://en.wikipedia.org/wiki/Internet_Archive,2026-04-20T20:16:49.112480+00:00,None,NaN


---
## 12. Compare pages before combining everything

Before reading the full combined dataset, it can help to compare how much material came from each page.

In [15]:
print("Links by source page:")
print(links_df["source_page"].value_counts(), "\n")

print("Images by source page:")
print(images_df["source_page"].value_counts(), "\n")

if "source_page" in sentences_df.columns and len(sentences_df) > 0:
    print("Sentences by source page:")
    print(sentences_df["source_page"].value_counts())
else:
    print("No sentence rows found.")

Links by source page:
source_page
https://en.wikipedia.org/wiki/Internet_Archive    2444
Name: count, dtype: int64 

Images by source page:
source_page
https://en.wikipedia.org/wiki/Internet_Archive    42
Name: count, dtype: int64 

Sentences by source page:
source_page
https://en.wikipedia.org/wiki/Internet_Archive    48
Name: count, dtype: int64


---
## 12. Read the combined dataset critically

Pause and ask:
- What appears more often: links, images, or sentences?
- What kinds of labels are missing?
- What context was removed by putting these into one table?
- What new patterns were created by combining them?

In [16]:
print(combined_df["item_type"].value_counts(dropna=False))
print("\nDomains:")
print(combined_df["domain"].dropna().value_counts())

item_type
link        2444
sentence      48
image         42
Name: count, dtype: int64

Domains:
domain
en.wikipedia.org            1872
web.archive.org              203
archive.org                   45
upload.wikimedia.org          36
blog.archive.org              33
                            ... 
lux.collections.yale.edu       1
developer.wikimedia.org        1
stats.wikimedia.org            1
www.wikimedia.org              1
www.mediawiki.org              1
Name: count, Length: 255, dtype: int64


---
## 13. Create a simple webpage to visualize the dataset

This step generates an HTML file you can open in a browser.

The page will:
- show each item as a card
- label whether it is a link, image, or sentence
- display thumbnails for image items when available
- link out to the source URL where relevant

In [17]:
cards = []

for _, row in combined_df.iterrows():
    label = str(row.get("label", "") or "")
    item_type = str(row.get("item_type", "") or "")
    absolute_url = str(row.get("absolute_url", "") or "")
    source_page = str(row.get("source_page", "") or "")
    thumb = row.get("thumbnail_url", None)

    image_html = ""
    if pd.notna(thumb) and thumb:
        image_html = f'<img src="{thumb}" alt="{label}" class="thumb">'

    card_html = f'''
    <div class="card">
      <div class="meta">{item_type}</div>
      {image_html}
      <div class="label">{label}</div>
      <div class="small"><strong>source page:</strong> <a href="{source_page}" target="_blank">{source_page}</a></div>
      <div class="small"><strong>item url:</strong> <a href="{absolute_url}" target="_blank">{absolute_url}</a></div>
    </div>
    '''
    cards.append(card_html)

html_page = f'''
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Week 4 Dataset Visualization</title>
  <style>
    body {{
      font-family: Arial, sans-serif;
      margin: 40px;
      background: #f7f7f7;
      color: #111;
    }}
    h1 {{
      font-size: 28px;
      margin-bottom: 8px;
    }}
    p {{
      max-width: 800px;
      line-height: 1.5;
    }}
    .grid {{
      display: grid;
      grid-template-columns: repeat(auto-fill, minmax(280px, 1fr));
      gap: 18px;
      margin-top: 28px;
    }}
    .card {{
      background: white;
      border: 1px solid #ddd;
      border-radius: 12px;
      padding: 14px;
      box-shadow: 0 2px 6px rgba(0,0,0,0.05);
      overflow-wrap: anywhere;
    }}
    .meta {{
      font-size: 12px;
      text-transform: uppercase;
      letter-spacing: 0.08em;
      color: #666;
      margin-bottom: 8px;
    }}
    .label {{
      font-weight: 600;
      margin: 10px 0;
    }}
    .small {{
      font-size: 12px;
      color: #444;
      margin-top: 6px;
    }}
    .thumb {{
      width: 100%;
      max-height: 180px;
      object-fit: cover;
      border-radius: 8px;
      border: 1px solid #eee;
    }}
  </style>
</head>
<body>
  <h1>Week 4 Dataset Visualization</h1>
  <p>
    This page visualizes a combined dataset made from scraped links and images.
    It does not preserve the original webpage interface. It reconstructs part of the page as a new archive.
  </p>
  <div class="grid">
    {"".join(cards)}
  </div>
</body>
</html>
'''

html_filename = "week4_dataset_visualization.html"
with open(html_filename, "w", encoding="utf-8") as f:
    f.write(html_page)

print(f"Saved webpage: {html_filename}")

Saved webpage: week4_dataset_visualization.html


---
## 14. Preview the webpage inside Colab

In [18]:
display(HTML(filename="week4_dataset_visualization.html"))

---
## 15. Download your outputs

The next cell makes clickable download links for:
- links dataset
- image dataset
- sentence dataset
- combined dataset
- webpage visualization

In [ ]:
display(FileLink(links_csv))
display(FileLink(images_csv))
display(FileLink(sentences_csv))
display(FileLink(combined_csv))
display(FileLink("week4_dataset_visualization.html"))

---
## 16. Assignment workflow

For your assignment:
1. choose 1–5 public pages
2. define one collection rule
3. collect 20–100 items total
4. save your links dataset
5. save your image dataset if relevant
6. save your sentence dataset if relevant
7. combine your datasets
8. generate a webpage to visualize them
9. write a short reflection

---
## 17. Reflection prompt

Write 300–500 words responding to these questions:

- What did you collect across the different pages?
- What did your method exclude?
- What changed when you combined the links, image, and sentence datasets?
- What differences emerged between pages?
- What is missing from the webpage visualization?
- Is this dataset an archive, a fragment, or a new artwork?
- If the pages you scraped disappeared tomorrow — taken down, blocked, or deleted — what would your dataset preserve? What would be gone that is not in your CSV?

In [ ]:
reflection = '''
Write your reflection here.
'''
print(reflection)

---
## 18. What this notebook cannot capture

A scraped dataset and a generated webpage usually do **not** preserve:
- the original interface
- user interaction
- temporal change
- recommendation systems
- platform behavior
- the full meaning of context
- deletion — if a page is taken down after you scrape it, your dataset may be the only record

This is why scraped archives are useful—but always incomplete.

---
## 18b. Optional: check if your pages exist in the Wayback Machine

The Internet Archive has archived over one trillion web pages.
But publishers including the New York Times, The Guardian, and Reddit
are now actively blocking its crawlers.

Run this cell to check whether your scraped pages have an archived snapshot —
and what happens when they do not.

If a page has no Wayback Machine snapshot and is taken down tomorrow,
your CSV may be the only copy.

In [19]:
print("Checking Wayback Machine availability...\n")

for page in pages:
    url = page["url"]
    api_url = f"http://archive.org/wayback/available?url={url}"
    try:
        r = requests.get(api_url, timeout=10)
        data = r.json()
        snapshot = data.get("archived_snapshots", {}).get("closest", {})
        if snapshot:
            print(f"found  {url}")
            print(f"  closest snapshot: {snapshot.get('timestamp', 'unknown')}")
            print(f"  status: {snapshot.get('status', 'unknown')}")
            print(f"  wayback url: {snapshot.get('url', 'unknown')}")
        else:
            print(f"not found  {url}")
            print(f"  no snapshot in the Wayback Machine")
            print(f"  your CSV may be the only copy of what you collected")
        print()
        time.sleep(1)  # one request per second — be polite
    except Exception as e:
        print(f"  could not check {url}: {e}")

print("If the pages you scraped were taken down tomorrow,")
print("would the Wayback Machine have them? Would your CSV?")
print("Which is more durable?")

Checking Wayback Machine availability...

found  https://archive.org
  closest snapshot: 20260411084013
  status: 200
  wayback url: http://web.archive.org/web/20260411084013/https://archive.org/

not found  https://en.wikipedia.org/wiki/Internet_Archive
  no snapshot in the Wayback Machine
  your CSV may be the only copy of what you collected

If the pages you scraped were taken down tomorrow,
would the Wayback Machine have them? Would your CSV?
Which is more durable?


---
## 19. Final takeaway

You are **not preserving the web as a whole**.  
You are creating a **partial, constructed archive** from it.

The datasets and the webpage are all new forms of representation.